# Restaurant Branch Performance — Exploratory Data Analysis (Day 13)

**Tools:** Pandas, NumPy, Matplotlib  
**Dataset:** `Day13_Restaurant_Branch_Performance_Dataset.csv`

### Objectives
- Inspect the structure and quality of the dataset.
- Generate summary statistics and descriptive analysis.
- Study distributions of important numerical variables.
- Compare performance across branches, regions, and store types.
- Examine relationships among numerical variables using correlation analysis.
- Finish with 5–8 data-driven observations.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Load the dataset
df = pd.read_csv("Day13_Restaurant_Branch_Performance_Dataset.csv")

# Display the first few records
df.head()

## 1. Dataset Overview

In [ ]:
print("Rows and columns:", df.shape)
print("\nColumn names:")
print(df.columns.tolist())

print("\nData types and non-null counts:")
df.info()

In [ ]:
# Check missing values
missing = df.isnull().sum().sort_values(ascending=False)
missing_percentage = (df.isnull().mean() * 100).sort_values(ascending=False)

missing_table = pd.DataFrame({
    "Missing_Count": missing,
    "Missing_Percentage": missing_percentage.round(2)
})
missing_table

In [ ]:
# Check duplicate rows
print("Duplicate rows:", df.duplicated().sum())

# Number of unique values in each column
df.nunique().sort_values()

**Data-quality note:** `Promotion` contains missing values. For analysis involving promotions, the missing category is treated as **No Promotion** rather than dropping those records. The other columns have no missing values.

## 2. Descriptive Statistics

In [ ]:
# Summary statistics for numerical columns
numeric_cols = df.select_dtypes(include="number").columns
df[numeric_cols].describe().T.round(2)

In [ ]:
# Summary statistics for categorical columns
categorical_cols = df.select_dtypes(include="object").columns
df[categorical_cols].describe().T

### Key business metrics

In [ ]:
metrics = {
    "Total Customers": df["Customers"].sum(),
    "Total Orders": df["Orders"].sum(),
    "Total Revenue": df["Revenue"].sum(),
    "Total Profit": df["Profit"].sum(),
    "Total Marketing Spend": df["Marketing_Spend"].sum(),
    "Average Customer Rating": df["Customer_Rating"].mean(),
    "Average Delivery Time (min)": df["Avg_Delivery_Min"].mean()
}

pd.Series(metrics).round(2)

## 3. Distribution Analysis

In [ ]:
# Histograms for important numerical variables
plot_cols = [
    "Customers", "Revenue", "Profit", "Marketing_Spend",
    "Staff_Count", "Avg_Delivery_Min", "Customer_Rating"
]

for col in plot_cols:
    plt.figure(figsize=(7, 4))
    plt.hist(df[col], bins=20, edgecolor="black")
    plt.title(f"Distribution of {col}")
    plt.xlabel(col)
    plt.ylabel("Frequency")
    plt.tight_layout()
    plt.show()

## 4. Categorical Analysis

In [ ]:
# Frequency distributions
for col in ["Branch", "Region", "Store_Type", "Weather", "Promotion"]:
    print(f"\n--- {col} ---")
    print(df[col].fillna("No Promotion").value_counts())

## 5. Branch Performance Comparison

In [ ]:
branch_summary = (
    df.groupby("Branch")
      .agg(
          Records=("Branch", "size"),
          Customers=("Customers", "mean"),
          Revenue=("Revenue", "mean"),
          Profit=("Profit", "mean"),
          Marketing_Spend=("Marketing_Spend", "mean"),
          Staff_Count=("Staff_Count", "mean"),
          Avg_Delivery_Min=("Avg_Delivery_Min", "mean"),
          Customer_Rating=("Customer_Rating", "mean")
      )
      .sort_values("Profit", ascending=False)
)

branch_summary.round(2)

In [ ]:
# Average profit by branch
plt.figure(figsize=(9, 5))
branch_summary["Profit"].sort_values().plot(kind="barh")
plt.title("Average Profit by Branch")
plt.xlabel("Average Profit")
plt.ylabel("Branch")
plt.tight_layout()
plt.show()

## 6. Region and Store-Type Comparison

In [ ]:
region_summary = (
    df.groupby("Region")
      .agg(
          Records=("Region", "size"),
          Customers=("Customers", "mean"),
          Revenue=("Revenue", "mean"),
          Profit=("Profit", "mean"),
          Marketing_Spend=("Marketing_Spend", "mean"),
          Avg_Delivery_Min=("Avg_Delivery_Min", "mean"),
          Customer_Rating=("Customer_Rating", "mean")
      )
      .sort_values("Profit", ascending=False)
)

store_summary = (
    df.groupby("Store_Type")
      .agg(
          Records=("Store_Type", "size"),
          Customers=("Customers", "mean"),
          Revenue=("Revenue", "mean"),
          Profit=("Profit", "mean"),
          Marketing_Spend=("Marketing_Spend", "mean"),
          Avg_Delivery_Min=("Avg_Delivery_Min", "mean"),
          Customer_Rating=("Customer_Rating", "mean")
      )
      .sort_values("Profit", ascending=False)
)

print("REGION SUMMARY")
display(region_summary.round(2))

print("\nSTORE TYPE SUMMARY")
display(store_summary.round(2))

In [ ]:
# Compare average profit by region
plt.figure(figsize=(7, 4))
region_summary["Profit"].plot(kind="bar")
plt.title("Average Profit by Region")
plt.xlabel("Region")
plt.ylabel("Average Profit")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

# Compare average profit by store type
plt.figure(figsize=(7, 4))
store_summary["Profit"].plot(kind="bar")
plt.title("Average Profit by Store Type")
plt.xlabel("Store Type")
plt.ylabel("Average Profit")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 7. Promotion Analysis

In [ ]:
df_analysis = df.copy()
df_analysis["Promotion"] = df_analysis["Promotion"].fillna("No Promotion")

promotion_summary = (
    df_analysis.groupby("Promotion")
    .agg(
        Records=("Promotion", "size"),
        Customers=("Customers", "mean"),
        Revenue=("Revenue", "mean"),
        Profit=("Profit", "mean"),
        Customer_Rating=("Customer_Rating", "mean")
    )
    .sort_values("Profit", ascending=False)
)

promotion_summary.round(2)

Promotion comparisons are descriptive only. A higher average revenue or profit for a promotion category does **not** by itself prove that the promotion caused the improvement, because other factors may differ between records.

## 8. Correlation Analysis

In [ ]:
# Pearson correlation matrix for numerical variables
corr_matrix = df[numeric_cols].corr().round(3)

corr_matrix

In [ ]:
# Correlations with Profit, ordered from strongest to weakest
profit_correlations = (
    corr_matrix["Profit"]
    .sort_values(ascending=False)
    .to_frame("Correlation_with_Profit")
)

profit_correlations

In [ ]:
# Correlation heatmap using Matplotlib
plt.figure(figsize=(12, 9))
plt.imshow(corr_matrix, aspect="auto")
plt.colorbar(label="Pearson Correlation")

plt.xticks(range(len(corr_matrix.columns)), corr_matrix.columns, rotation=90)
plt.yticks(range(len(corr_matrix.index)), corr_matrix.index)

plt.title("Correlation Matrix of Numerical Variables")
plt.tight_layout()
plt.show()

### Interpreting the correlation matrix

- Values close to **+1** indicate a strong positive linear relationship.
- Values close to **−1** indicate a strong negative linear relationship.
- Values close to **0** indicate little or no linear relationship.
- Correlation measures association, **not causation**.

## 9. Additional Relationship Checks

In [ ]:
# A few useful pairwise relationships
pairs = [
    ("Customers", "Revenue"),
    ("Average_Bill", "Revenue"),
    ("Marketing_Spend", "Revenue"),
    ("Avg_Delivery_Min", "Customer_Rating"),
    ("Customer_Rating", "Profit")
]

for x, y in pairs:
    plt.figure(figsize=(6, 4))
    plt.scatter(df[x], df[y], alpha=0.6)
    plt.title(f"{x} vs {y}")
    plt.xlabel(x)
    plt.ylabel(y)
    plt.tight_layout()
    plt.show()

## 10. Final Data-Driven Observations

- 1. The dataset contains 350 restaurant records and 22 columns. The Promotion column has 184 missing values, while the other columns are complete.
- 2. Profit is most strongly associated with Revenue (Pearson r = 0.967), followed by Food Cost (r = 0.918), Operating Cost (r = 0.862), Customers (r = 0.836), and Average Bill (r = 0.828). These are associations, not proof of causation.
- 3. Bengaluru has the highest average branch profit at ₹31,891, while Mumbai has the lowest at ₹26,581.
- 4. The South region has the highest average profit (₹30,691), slightly ahead of the North (₹30,626); the West is lowest (₹28,721).
- 5. Premium stores have much higher average customers (298.7) and profit (₹62,890) than Standard and Express stores.
- 6. Average customer rating is 4.47/5 and has almost no linear relationship with profit (r = -0.005), suggesting that ratings alone do not explain financial performance in this dataset.
- 7. Loyalty Offer records show the highest average revenue (₹91,938) and profit (₹32,346) among the promotion categories represented, while No Promotion records have lower averages.

## Conclusion

The EDA shows clear differences in restaurant performance across branches, regions, and store types. Revenue, customer volume, orders, average bill, and cost variables show strong relationships with profit, while customer rating and delivery time have much weaker linear relationships with profit. Premium stores stand out with substantially higher average customer volume and profitability. These findings provide useful descriptive insights for further business analysis, but causal conclusions would require additional statistical testing and experimental or longitudinal evidence.